# CSP

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/11-csp/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍與後續實驗〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.4.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
import torch
from torch import nn


class CSP(nn.Module):
    def __init__(self):
        super().__init__()
        self.branch = nn.Sequential(nn.Conv2d(4,4,3,padding=1),nn.ReLU(),
                                    nn.Conv2d(4,4,3,padding=1),nn.ReLU())
        self.fuse = nn.Conv2d(8,8,1)

    def forward(self,x):
        bypass, transformed = x.chunk(2,dim=1)
        return self.fuse(torch.cat([bypass,self.branch(transformed)],dim=1))


class Full(nn.Module):
    def __init__(self):
        super().__init__()
        self.branch = nn.Sequential(nn.Conv2d(8,8,3,padding=1),nn.ReLU(),
                                    nn.Conv2d(8,8,3,padding=1),nn.ReLU())
        self.fuse = nn.Conv2d(8,8,1)

    def forward(self,x):
        return self.fuse(self.branch(x))


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    a = torch.tensor([[[[1.]],[[2.]]]])
    b = torch.tensor([[[[10.]],[[20.]]]])
    joined, added = torch.cat([a,b],dim=1), a+b
    assert joined.shape == (1,4,1,1) and joined.flatten().tolist() == [1,2,10,20]
    assert added.shape == (1,2,1,1) and added.flatten().tolist() == [11,22]
    print('concat',tuple(joined.shape),joined.flatten().tolist(),
          'add',tuple(added.shape),added.flatten().tolist())
    full,csp = Full(),CSP()
    counts = [sum(p.numel() for p in m.parameters()) for m in [full,csp]]
    assert counts == [1240,368]
    for name,model in [('full',full),('CSP',csp)]:
        x = torch.randn(2,8,8,8,requires_grad=True)
        optimizer = torch.optim.SGD(model.parameters(),lr=.01)
        before = model.fuse.weight.detach().clone()
        output = model(x)
        assert output.shape == x.shape
        loss = output.square().mean()
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        assert torch.isfinite(x.grad).all()
        assert all(p.grad is not None and torch.isfinite(p.grad).all() and p.grad.abs().sum() > 0
                   for p in model.parameters())
        assert x.grad[:,:4].abs().sum() > 0
        assert x.grad[:,4:].abs().sum() > 0
        optimizer.step()
        assert not torch.equal(before,model.fuse.weight.detach())
        print(name,'input gradient abs sums first4/last4',
              round(x.grad[:,:4].abs().sum().item(),6),round(x.grad[:,4:].abs().sum().item(),6))
        print(name,'output',tuple(output.shape),'finite backward and step')
    print('parameters full/CSP',counts,'capacity differs; no accuracy comparison')


if __name__ == '__main__':
    main()


concat (1, 4, 1, 1) [1.0, 2.0, 10.0, 20.0] add (1, 2, 1, 1) [11.0, 22.0]
full input gradient abs sums first4/last4 0.033093 0.029087
full output (2, 8, 8, 8) finite backward and step
CSP input gradient abs sums first4/last4 0.255476 0.019409
CSP output (2, 8, 8, 8) finite backward and step
parameters full/CSP [1240, 368] capacity differs; no accuracy comparison
